In [15]:
import pandas as pd
import re

In [ ]:
sales = pd.read_csv(r"C:\Users\ReMarkt\Documents\MDDB\database project\data\raw\sales.csv")


In [13]:
movies = pd.read_csv(r"C:\Users\ReMarkt\Documents\MDDB\database project\data\cleaned\movie_table.csv")

In [12]:
sales["movie_name"] = sales["url"].str.split("/", expand=True).iloc[:,-1]

In [31]:
stop_words = {"the", "a", "an", "of", "and", "in", "to", "for"}
def normalized_movie_key(name, stopwords):
    name = str(name).lower()
    name = re.sub(r'\s*[\(\[].*[a-zA-Z].*[\)\]]', "", name)
    name = name.replace("+", "")
    name = re.sub(r"[^\w\s]", " ", name)
    words = sorted([word for word in name.split() if word not in stop_words])
    return " ".join(words)

In [ ]:
sales["movie_key"] = sales["movie_name"].apply(lambda x: normalized_movie_key(x, stopwords=stop_words))


In [32]:
movies["movie_key"] = movies["movie_name"].apply(lambda x: normalized_movie_key(x, stopwords=stop_words))

In [35]:
sales_df = sales.merge(movies[["movie_key", "movie_id"]], on="movie_key", how="left")

In [42]:
sales_df["sales_id"] = sales_df.index +1

In [43]:
sales_df.head()

,year,release_date,title,genre,international_box_office,domestic_box_office,worldwide_box_office,production_budget,Unnamed: 8,opening_weekend,theatre_count,avg run per theatre,runtime,keywords,creative_type,url,movie_name,movie_key,movie_id,sales_id
0,2000,January 1st,Bakha Satang,Drama,76576.0,NaN,76576.0,NaN,NaN,NaN,NaN,NaN,129.0,NaN,Contemporary Fiction,https://www.the-numbers.com/movie/Bakha-Satang...,Bakha-Satang-(S-Korea),bakha satang,1.0,1
1,2001,January 12th,Antitrust,Thriller/Suspense,6900000.0,10965209.0,17865209.0,30000000.0,NaN,5486209.0,2433.0,3.1,NaN,NaN,Contemporary Fiction,https://www.the-numbers.com/movie/Antitrust,Antitrust,antitrust,2.0,2
2,2000,January 28th,Santitos,NaN,NaN,378562.0,NaN,NaN,NaN,NaN,NaN,NaN,105.0,NaN,NaN,https://www.the-numbers.com/movie/Santitos,Santitos,santitos,3.0,3
3,2002,2002 (Wide) by,Frank McKlusky C.I.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.the-numbers.com/movie/Frank-McKlus...,Frank-McKlusky-C-I,c frank i mcklusky,4.0,4
4,2002,January 25th,A Walk to Remember,Drama,4833792.0,41227069.0,46060861.0,11000000.0,NaN,12177488.0,2411.0,5.3,NaN,Coming of Age,Contemporary Fiction,https://www.the-numbers.com/movie/Walk-to-Reme...,Walk-to-Remember-A,remember walk,5.0,5


In [45]:
sales_df = sales_df.rename(columns={"avg run per theatre": "avg_run_per_theatre"})

In [48]:
sales_table = sales_df[['sales_id', 'movie_id', 'production_budget', 'domestic_box_office',
       'international_box_office', 'worldwide_box_office', 'theatre_count',
       'avg_run_per_theatre']]

In [49]:
sales_table

,sales_id,movie_id,production_budget,domestic_box_office,international_box_office,worldwide_box_office,theatre_count,avg_run_per_theatre
0,1,1.0,NaN,NaN,76576.0,76576.0,NaN,NaN
1,2,2.0,30000000.0,10965209.0,6900000.0,17865209.0,2433.0,3.1
2,3,3.0,NaN,378562.0,NaN,NaN,NaN,NaN
3,4,4.0,NaN,NaN,NaN,NaN,NaN,NaN
4,5,5.0,11000000.0,41227069.0,4833792.0,46060861.0,2411.0,5.3
...,...,...,...,...,...,...,...,...
30607,30608,30215.0,NaN,NaN,12356.0,12356.0,NaN,NaN
30608,30609,30216.0,NaN,54730.0,914119.0,968849.0,123.0,2.0
30609,30610,30217.0,NaN,NaN,240000.0,240000.0,NaN,NaN
30610,30611,30218.0,NaN,364397.0,16987526.0,17351923.0,186.0,2.5


In [51]:
sales_table["total_revenue"] = sales_table["domestic_box_office"]+sales_table["international_box_office"]

In [54]:
sales_table["movie_id"] = sales_table["movie_id"].astype("Int64")

In [55]:
sales_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 30612 entries, 0 to 30611
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   sales_id                  30612 non-null  int64  
 1   movie_id                  30606 non-null  Int64  
 2   production_budget         4480 non-null   float64
 3   domestic_box_office       11884 non-null  float64
 4   international_box_office  21575 non-null  float64
 5   worldwide_box_office      21575 non-null  float64
 6   theatre_count             10963 non-null  float64
 7   avg_run_per_theatre       10952 non-null  float64
 8   total_revenue             6934 non-null   float64
dtypes: Int64(1), float64(7), int64(1)
memory usage: 2.1 MB


In [60]:
sales_table.shape

(30612, 9)

In [63]:
sales_table = sales_table.drop_duplicates(subset="movie_id")

In [64]:
sales_table.shape

(30219, 9)

In [65]:
sales_table.to_csv(r"C:\Users\ReMarkt\Documents\MDDB\database project\data\cleaned\sales_table.csv", index=False)